# Noms du binome: 

FERNANDES--STRAG, TORRES--LEGUET

# Prénoms du binome:

Henry, Alexandre

## Note: exporter le compte rendu basé sur le notebook au format pdf


# Entropie et génération de mots de passe
* L'objectif de ce tp est de continuer à se familiariser avec la notion d'entropie, mais aussi de comprendre le lien qu'il existe entre cette mesure informationelle et la sécurité d'un générateur (humain ou executable) de mots de passes
* Ainsi, nous proposons d'étudier l'entropie d'un tel générateur, et ce en fonction du modèle probabiliste considéré pour le modéliser (contruit à partir d'une lettre, de deux lettres, de 4 lettres, ...). A l'aide de tirrages aléatoires, nous estimerons également le temps moyen nécessaire pour trouver un mot de passe à partir de ce modèle.
* A la fin de ce TP, nous considérerons un générateur de mots de passe spécifique qui générera un mot de passe en prennant **4 lettres consécutives dans un texte** (sans se soucier des espaces). Ces lettres peuvent faire parti d'un ou de plusieurs mots consécutifs.
* Nous faisons l'hypothèse que le texte n'est composé que des 26 lettres de l'alphabet, sans majuscules ni accents

Nous chercherons aussi à comprendre (voir dernière question):
- les bonnes pratiques pour le défenseur, i.e. la personne cherchant à générer/construire un système de génération de mots de passe.
- les bonnes pratiques pour l'attaquant, i.e. la personne essayant de trouver le mot de passe.

**Il est important de commenter vos réponses, en utilisant des cellules markdown**


In [230]:
import numpy as np

from numpy import genfromtxt
from pandas import read_csv
import pandas as pd
import time

#### Modèle monogramme (une lettre) : le générateur génère des mots de passe à partir des occurences des monogrammes
* On récupére des données composées de [lettre,frequence d'apparition de la lettre] (voir fichier csv pour [comma-separated-value](https://en.wikipedia.org/wiki/Comma-separated_values)) 

In [231]:
monogramme = read_csv('monogramme.csv')
freq_mono = (monogramme['frequency']).values
letters_mono = (monogramme['letters']).values

Q: Quelles sont les 5 lettres les plus représentées ?

In [232]:
letters_mono[0:5]

<StringArray>
['E', 'S', 'A', 'N', 'T']
Length: 5, dtype: str

On remarque que le fichier 'monogramme.csv' a déjà trié les lettres par leur fréquence.
Il suffit donc juste de récupérer les 5 premiers éléments de la liste letters_mono : E,S,A,N,T.
Cela correspond bien aux lettres les plus fréquentes dans un texte écrit en français. 

Ecrire une fonction qui calcule l'entropie à partir d'un vecteur constitué de probabilités empiriques (note, il est important de bien *gérer* le cas ou la probabilité est nulle).

In [233]:
def entropie(freq):
    ent = 0
    for f in freq:
        if f > 0 and f < 1:
            ent += f * np.log2(f)
    return -ent

In [234]:
entropie([0.95, 0.05, 0])

np.float64(0.28639695711595625)

Q: en utilisant ce modèle probabiliste pour générer un mot de passe, quelle est l'entropie d'un mot de passe de 8 lettres ?

In [235]:
ent = entropie(freq_mono)
ent_totale = ent * 8

print(ent)
print(ent_totale)

3.959530303722292
31.676242429778338


Pour un mot de passe de 8 lettres généré en tirant aléatoirement et indépendamment des lettres selon la distribution lettres_mono, on peut simplement multiplier par 8 l'entropie de la distribution du monogramme, grâce à la propriété :
$H(X,Y) = H(X) + H(Y)$ où $X$ et $Y$ sont indépendantes (ici, les variables aléatoires indépendantes sont les tirages de chaque lettre). On obtient ainsi une entropie de 31.67.

Cela revient à tirer au hasard et uniformément un mot de passe parmi 2^(31.68) valeurs possibles, soit environ 3,4 milliards. (ie, la distribution de probabilité uniforme sur un ensemble de 3,4 milliards de valeurs a cette même entropie de 31.67)

Q: A l'aide de la fonction `np.random.choice()`, estimer le temps nécessaire en secondes pour tirer 100 000 mots de passes en utilisant ce générateur ? (note: ici le tirage n'est pas forcemment réaliste, car aléatoire, mais l'idée est surtout de mesurer le temps minimal nécessaire pour générer N mots de passes).

In [236]:
np.random.choice(letters_mono, 10, p=freq_mono)

array(['M', 'D', 'C', 'A', 'D', 'A', 'T', 'A', 'E', 'N'], dtype=object)

In [237]:
times = 0
N = 100
N_pwd = 100000

for _ in range(N):
    t = time.time()
    for _ in range(8):
        np.random.choice(letters_mono, N_pwd, p=freq_mono)
    t_100000 = time.time() - t
    times += t_100000

print(times/N, 's')

0.014838182926177978 s


On peut générer 100 000 mots de passe en beaucoup moins d'une seconde sur un ordinateur portable.

Nous definissons l'"entropie du devin" G (guessing entropie) comme le **nombre moyen d'essais successif nécessaires pour trouver un mot de passe à partir de notre générateur**.
    On peut montrer que $G\geq 2^H/4+1$ où $H$ est l'entropie de la source (voir le papier Password_Entropy_and_Password_Quality.pdf )

Q: calculer le minorant de $G$ pour ce modèle

In [238]:
inf = np.pow(2, ent_totale) / 4 + 1

print(inf)

857904864.6814479


On obtient une borne basse qui vaut 860 millions.
Cela veut dire que le nombre moyens d'essais successifs pour trouver le mot de passe est plus grand que 857M.

On peut comparer ce nombre avec le nombre de 3,4 milliards trouvé plus haut (équivalent à l'entropie de 31.68) : c'est environ le quart.

Q: combien de temps cela prendra-t-il pour trouver un mot de passe si l'on suppose qu'il est possible de prendre le générateur codé précédemment ? (en minutes)

In [239]:
time_per_pwd = (times/N) / N_pwd

print(time_per_pwd, 's')
print(time_per_pwd * inf, 's')
print(time_per_pwd * inf / 60, 'min')

1.483818292617798e-07 s
127.2974931540129 s
2.121624885900215 min


La borne basse est donc de 127 secondes environ. Cela veut dire qu'en quelques minutes de recherche, on peut tomber sur le mot de passe.

On propose maintenant d'utiliser un modèle plus évolué qui est construit à partir de la probabilité conjointe de deux lettres successives (bigramme)

In [240]:
bigramme = read_csv('bigramme.csv',keep_default_na=False)

#print(bigramme.head(5))
freq_bi = (bigramme['frequency']).values

letters_bi = (bigramme['letters']).values

Q: Quelles sont les 5 couples de lettres les plus représentés ?

In [241]:
bigramme.sort_values(by='frequency', ascending=False, inplace=True)
bigramme.head(5)

,letters,frequency
122,ES,0.023809
117,EN,0.021248
82,DE,0.019570
290,LE,0.018845
357,NT,0.017009


Q: en utilisant ce modèle probabiliste pour générer un mot de passe, quelle est l'entropie d'un mot de passe de 8 lettres ?

In [242]:
ent = entropie(freq_bi)
ent_totale = 4 * ent

print(ent)
print(ent_totale)

7.535566011616047
30.142264046464188


On utilise la même méthode de calculs que précédemment, en sommant les entropies des variables indépendantes (ici, le tirage d'un bigramme)

Q: Pourquoi cette entropie est-elle inférieure à celle du modèle construit sur des monogrammes ? Quelle propriété théorique de l'entropie peut justifier ce constat  ?

La baisse d'entropie est expliquée par cette formule :

$$
H(X,Y) = H(X) + H(Y) - I(X,Y)
$$

qui, appliquée à nos 8 variables :

$$
H(X_1,...,X_8) = \Sigma_i H(X_i) - \Sigma_{i \neq j} I(X_i, X_j)
$$

par construction des mots de passe, les informations mutuelles I(X_1, X_3), I(X_3,X_5) etc (ie, les lettres qui ne sont pas tirés en même temps) sont nulles. Il reste seulement :

$$
H(X_1,...,X_8) = \Sigma_i H(X_i) - \Sigma_{i,j \in \text{bigramme}} I(X_i, x_j)
$$

On voit donc que $H(X_1,...,X_8)$ est bien plus petit que $\Sigma_i H(X_i)$, et cela est dû au fait qu'on perd l'information mutuelle au sein des lettres tirées en même temps.

En effet, en français, deux lettres successives ne sont pas indépendantes (par exemple, Q est presque toujours suivi de U), donc $I(X;Y) > 0$. L'entropie d'un bigramme est donc strictement inférieure à deux fois l'entropie d'une lettre, et un mot de passe de 8 lettres construit à partir de 4 bigrammes a une entropie plus faible (30,1 bits) que celui construit à partir de 8 lettres indépendantes (31,7 bits). Le modèle bigramme capture une partie de la structure de la langue, ce qui rend les mots de passe plus prévisibles.

Q: A l'aide de la fonction `np.random.choice()`, calculer le temps nécessaire en secondes pour tirer 100 000 mots de passes en utilisant ce générateur ?

In [243]:
times = 0
N = 100
N_pwd = 100000

for _ in range(N):
    t = time.time()
    for _ in range(4):
        np.random.choice(letters_bi, N_pwd, p=freq_bi)
    t_100000 = time.time() - t
    times += t_100000

print(times/N, 's')

0.016344306468963624 s


Q: calculer le minorant de $G$ pour ce modèle

In [244]:
inf = np.pow(2, ent_totale) / 4 + 1

print(inf)

296254956.70154107


Q: combien de temps cela prendra-t-il pour trouver un mot de passe si l'on suppose qu'il est possible de prendre le générateur codé précédemment ? (en minutes)

In [245]:
time_per_pwd = (times/N) / N_pwd

print(time_per_pwd, 's')
print(time_per_pwd * inf, 's')
print(time_per_pwd * inf / 60, 'min')

1.6344306468963624e-07 s
48.42081805279536 s
0.807013634213256 min


Ce mot de passe est plus facile à trouver (ce qui est normal car son entropie est plus faible).

Q: **Modèle Uniforme:** si maintenant on change de stratégie et on tire aléatoirement chaque lettre de l'alphabet de façon uniforme, quelle est l'entropie de ce nouveau générateur ?

In [246]:
ent = entropie([1/26]*26)
ent_totale = 8 * ent

print(ent)
print(ent_totale)

4.70043971814109
37.60351774512872


On trouve 37.6. Comme on est dans le cas d'une distribution uniforme sur 26^8 valeurs, on pouvait aussi directement calculer log2(26^8)=37.6.

Q: A l'aide de la fonction `np.random.choice()`, calculer le temps nécessaire en secondes pour tirer 100 000 mots de passes en utilisant ce générateur ?

In [247]:
times = 0
N = 100
N_pwd = 100000
proba = [1/26]*26

for _ in range(N):
    t = time.time()
    for _ in range(8):
        np.random.choice(letters_mono, N_pwd, p=proba)
    t_100000 = time.time() - t
    times += t_100000

print(times/N, 's')

0.018571557998657225 s


Q: calculer le minorant de $G$ pour ce modèle

In [248]:
inf = np.pow(2, ent_totale) / 4 + 1

print(inf)

52206766144.99937


Q: dans ce cas précis, quelle est la valeur exacte de $G$?

In [249]:
G = np.pow(26, 8)

print(G)

208827064576


<b>Cas avec remise (naif) :</b>

Dans ce cas, il y a 26^8 possibilités équiprobables, et chaque essai consiste à tirer au hasard parmi toutes ces possibilités. (les tirages sont avec remise et indépendants). L'évènement "trouver le vrai mot de passe" suit donc une loi géométrique (loi du premier succès) de paramètre p=26^(-8), d'espérance 1/p = 26^8 = 208 milliards.

<b>Cas sans remise:</b>
Notons $N = 26^8$ le nombre de mots de passe possibles de 8 lettres, et $w_1, \dots, w_N$ ces mots. Soit $X$ la variable aléatoire « mot de passe produit par le générateur uniforme ». Les lettres sont tirées indépendamment et uniformément, donc pour tout $i \in \{1, \dots, N\}$ :
$$P(X = w_i) = \left(\frac{1}{26}\right)^8 = \frac{1}{N}.$$

L'attaquant choisit un ordre de test, c'est-à-dire une permutation $\sigma$ de $\{1, \dots, N\}$. Il teste $w_{\sigma(1)}$, puis $w_{\sigma(2)}$, etc., sans jamais retester un mot (sinon il gaspille un essai).

Soit $T$ le nombre d'essais nécessaires pour trouver le mot de passe :
$$T = k \iff X = w_{\sigma(k)}, \qquad k \in \{1, \dots, N\}.$$
On a donc, pour tout $k \in \{1, \dots, N\}$ :
$$P(T = k) = P\big(X = w_{\sigma(k)}\big) = \frac{1}{N}.$$
$T$ suit la loi uniforme sur $\{1, \dots, N\}$, et ce quel que soit l'ordre $\sigma$.

$$G = E[T] = \sum_{k=1}^{N} k \, P(T = k) = \frac{1}{N} \sum_{k=1}^{N} k = \frac{1}{N} \cdot \frac{N(N+1)}{2} = \frac{N+1}{2}.$$

Pour une loi quelconque, $E[T] = \sum_k k \, P(X = w_{\sigma(k)})$ est minimale quand les mots sont testés par probabilité décroissante (inégalité de réarrangement : on associe les plus petits $k$ aux plus grandes probabilités). Dans le cas uniforme, toutes les probabilités sont égales, donc tous les ordres sont optimaux et donnent $E[T] = \frac{N+1}{2}$. C'est bien la valeur exacte de $G$ :
$$G = \frac{26^8 + 1}{2} \approx 1{,}04 \times 10^{11}$$

$H = 8 \log_2 26 = \log_2 N$, donc $2^H/4 + 1 = N/4 + 1$, et on a bien $G = \frac{N+1}{2} \geq \frac{N}{4} + 1$ (soit environ 2 fois le minorant).


Q: combien de temps cela prendra-t-il pour trouver un mot de passe en utilisant le générateur codé précédemment ? (en minutes)

In [250]:
time_per_pwd = (times/N) / N_pwd

print(time_per_pwd, 's')
print(time_per_pwd * G, 's')
print(time_per_pwd * G / 60, 'min')

1.8571557998657226e-07 s
38782.439414625216 s
646.3739902437536 min


Q: implémenter une attaque pratique qui consiste à:
1. **pour le défenseur: (la personne qui génère le mot de passe)** tirer un mot de passe de 4 lettres consécutives à partir de ce texte de Victor Hugo (texteFrancais.txt) tiré des Misérables.  
2. **pour l'attaquant:** utiliser le modèle bigramme pour générer des mots de passe et minimiser le nombre d'essais. Pour cela on pourra :
    * dans un premier temps pré-calculer un **dictionnaire**, qui contriendra un nombre de MdP générés classés dans l'ordre du plus probable au moins probable et qui ne contient pas de doublons 
    * dans un deuxième temps appeler ce dictionnaire pour comparer chacune de ses entrées au mot de passe généré.
3. Il faudra faire ses tests plusieurs fois afin de d'obtenir un nombre moyens d'appel au dictionnaire nécessaire
4. Il sera intéressant de comparer le nombre trouvé à la valeur de G (qui est une borne inférieure)
5. Question annexe: Par un simple calcul, si le générateur utilisé n'est plus ce générateur mais un générateur qui tire chaque lettre de façon équiprobable, rappeler la valeur de G. Comparer cette valeur avec la valeur trouvée en utilisant la stratégie "des 4 lettres consécutives".

In [251]:
## Fonction générant un mot de passe
def get_passwd():
    text_hugo = open("texteFrancais.txt","r")
    str_hugo = str(text_hugo.read())

    # On remplace des lettres avec accent avec des lettres sans accent
    str_hugo = str_hugo.replace("Â", "A")
    str_hugo = str_hugo.replace("Ù", "U")
    str_hugo = str_hugo.replace("Ô", "O")
    size_txt = len(str_hugo)

    idx_rand = np.random.randint(size_txt-4)
    #print(idx_rand)

    psswd = str_hugo[idx_rand:idx_rand+4]
    return(psswd)

In [252]:
N_dict = 1_000_000
proba_bi = freq_bi

# on génère N_dict mot de passe de 4 lettres à partir des b1 et b2
b1 = np.random.choice(letters_bi, N_dict, p=proba_bi).astype(str)
b2 = np.random.choice(letters_bi, N_dict, p=proba_bi).astype(str)
# qu'on place dans une liste (np.char.add pour concat des tableaux de string)
tab_passwd = np.char.add(b1, b2)

print(tab_passwd)

['ITTQ' 'AVIT' 'EFEE' ... 'NAAL' 'ORSI' 'LHNT']


In [ ]:
# on récupère la liste des mdps uniques et leur nombre (pour estimer leur proba)
mdps, counts = np.unique(tab_passwd, return_counts=True)

ordre = np.argsort(-counts) # indices des mdps triés par ordre décroissant de fréquence

dico = mdps[ordre]
freq_dico = counts[ordre] / N_dict # calcul de la proba empirique

print(dico)
print(freq_dico)

# une autre approche est de considérer tous les couples de bigrammes possibles (676^2)
# pour chacun des couples, on peut calculer leur probabilité grâce à proba_bi
# mais cette méthode n'est plus envisageable avec des modèles trigrammes, etc, alors que la présente méthode oui

['ESES' 'ENES' 'ESEN' ... 'RQAL' 'RQAB' 'ZYTO']
[5.80e-04 5.23e-04 5.17e-04 ... 1.00e-06 1.00e-06 1.00e-06]


In [254]:
pd.DataFrame({'mdp': dico[:10], 'frequence': freq_dico[:10]})

,mdp,frequence
0,ESES,0.000580
1,ENES,0.000523
2,ESEN,0.000517
3,ESLE,0.000497
4,ENEN,0.000457
5,LEES,0.000456
6,ESDE,0.000450
7,DEES,0.000440
8,ESNT,0.000430
9,ENDE,0.000425


In [255]:
NB_ATTAQUES = 1000
essais_moyens = 0

for _ in range(NB_ATTAQUES):
    pwd = get_passwd() # mot de passe caché, à trouver

    essais = 0
    for i, m in enumerate(dico): # dico est rangé du plus probable au moins probable
        essais += 1
        if m == pwd:
            break

    essais_moyens += essais

print(essais_moyens / NB_ATTAQUES)

13772.467


Le nombre d'essais moyen (ou de façon équivalente le nombre d'appels au dictionnaire) est d'environ 13k.
Par ailleurs, pour un attaquant qui utilise le modèle bigramme, la borne inf de G vaut ici :

In [256]:
ent = 2 * entropie(freq_bi)
G_inf = np.power(2, ent) / 4 + 1

print(ent)
print(G_inf)

15.071132023232094
8607.029219412707


On trouve environ 13 000 essais en moyenne, contre une borne inférieure de G d'environ 8 600. Le résultat est donc cohérent.

Si le générateur tire chaque lettre de façon équiprobable, il y a $26^4$ mots de passe équiprobables, soit G qui vaut le nombre de possibilités:

In [257]:
G_unif = 26**4
print(G_unif)

456976


Avec un générateur uniforme, il faut environ 460k essais en moyenne, contre environ 13 000 avec la stratégie des 4 lettres consécutives.

## Conclusions 

- Définir des bonnes pratiques pour le défenseur, i.e. la personne cherchant à concevoir un système de génération de mots de passe ? 
- Définir des bonnes pratiques pour l'attaquant, i.e. la personne essayant de trouver le mot de passe ?

**Pour le défenseur :**
- tirer les caractères au hasard/uniformément plutôt que de les prendre dans un texte ou bien d'utiliser des mots courants (la structure de la langue fait baisser l'entropie et donc est plus facile à deviner)
- prendre des mots de passe longs, avec un alphabet plus grand (majuscules, chiffres, symboles), car G augmente exponentiellement avec l'entropie.

**Pour l'attaquant :**
- utiliser un modèle qui ressemble à la façon dont les mots de passe sont réellement générés (n-grammes de la langue, mots courants, fuites de mots de passe etc)
- tester les mots de passe du plus probable au moins probable à l'aide d'un dictionnaire pré-calculé grâce au point précédent.

## Un peu de lecture
Cet article montre comment des hackers, à partir de leaks de bases de mots de passes, peuvent rapidement arriver à trouver le votre:
https://arstechnica.com/information-technology/2013/05/how-crackers-make-minced-meat-out-of-your-passwords/
